# Full benchmark: Cui-inspired TL and RL patience sensitivity

Runs **RNN, LSTM and GRU × 15, 30, 45 and 60 minutes × seeds 41–43**, all 12 patients.
This is the full-grid counterpart of `run_schedule_sensitivity_on_colab.ipynb`,
with the training, combined seed results, and every stage of the main analysis workflow.

| Arm | RL | Shared TL |
|---|---|---|
| patience5 | 200-epoch cap, patience 5 | 10 epochs at 0.0003 + 10 at 0.00005 |
| patience15 | 200-epoch cap, patience 15 | same fitted models |
| fixed20 | 20 epochs, no early stopping | same fitted models |

RL uses a **single stage** starting at 0.0003 in all arms. TL resets Adam between stages
and always completes 10+10 epochs. All models use 128 hidden units, 2 layers, dropout 0.2,
batch 16, four inputs and a 60-minute history. All arms retain validation-based scheduling
and best-checkpoint selection. This adapts Cui's stage schedule to our recurrent models;
it does not reproduce Cui's architecture or test-based checkpoint selection.

**This is a new full experiment, not a reproduction of the article's current numbers.**
Do not mix its outputs with prior runs. The code fingerprints all source/config/data inputs.
There are 144 mode/seed jobs: 36 shared TL and 108 RL. Each job covers 12 patients.
A completed job is mirrored to Drive; an interrupted job repeats at most that mode/seed job.
Analysis runs for each arm, so this takes more time and Drive space than the original grid.

Commit/push this notebook, `RUN/experiments/run_cui_full_grid.py`, and the updated
`benchmark/configs/config_manager.py` and `benchmark/experiments/configured.py` to `bench2`.


## 1. Drive and settings

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
DRIVE_DATA = Path('/content/drive/MyDrive/ohiot1dm')
DRIVE_RESULTS = Path('/content/drive/MyDrive/bg-results')
REPO_DIR = Path('/content/BG-forecasting-cui-full-grid')
REPO_URL = 'https://github.com/beatriz-fulgencio/BG-forecasting.git'
BRANCH = 'bench2'
MODELS = ['gru', 'lstm', 'rnn']
HORIZONS = [15, 30, 45, 60]
SEEDS = [41, 42, 43]
SMOKE_TEST = False  # True: two patients, GRU/30, one seed, tiny model; not paper evidence
REPLICATES = 20000
PERMUTATIONS = 10000


## 2. Checkout and dependencies

In [ ]:
import os, sys, subprocess, shutil, importlib.util, json
if not REPO_DIR.exists():
    subprocess.run(['git','clone','--branch',BRANCH,REPO_URL,str(REPO_DIR)],check=True)
else:
    subprocess.run(['git','-C',str(REPO_DIR),'pull','--ff-only'],check=True)
os.chdir(REPO_DIR)
subprocess.run([sys.executable,'-m','pip','install','-q','-e','.'],check=True)
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
import torch
if not torch.cuda.is_available() and not SMOKE_TEST:
    raise RuntimeError('Select a GPU runtime before running the full grid.')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
helper = REPO_DIR / 'RUN/experiments/run_cui_full_grid.py'
if not helper.exists():
    raise FileNotFoundError('Push run_cui_full_grid.py to the selected branch first.')
sys.path.insert(0, str(REPO_DIR))
spec = importlib.util.spec_from_file_location('cui_grid', helper)
grid = importlib.util.module_from_spec(spec);spec.loader.exec_module(grid)
print('Device:', torch.cuda.get_device_name(0) if DEVICE == 'cuda' else 'CPU smoke')


## 3. Stage and verify the dataset

In [ ]:
COHORT = {'2018':[559,563,570,575,588,591], '2020':[540,544,552,567,584,596]}
for release, patients in COHORT.items():
    for pid in patients:
        if SMOKE_TEST and pid not in [559,563]:
            continue
        for mode in ['train','test']:
            name = f'{pid}-ws-{mode}ing.xml'
            source = DRIVE_DATA / release / mode / name
            target = REPO_DIR / 'data/raw/ohiot1dm' / release / mode / name
            if not source.is_file():raise FileNotFoundError(source)
            target.parent.mkdir(parents=True, exist_ok=True)
            if not target.is_file() or grid.sha(source) != grid.sha(target):
                shutil.copy2(source,target)
            assert grid.sha(source) == grid.sha(target)
print('Raw inputs verified.')


## 4. Freeze the protocol and restore completed jobs

In [ ]:
if SMOKE_TEST:
    MODELS, HORIZONS, SEEDS = ['gru'], [30], [41]
    REPLICATES, PERMUTATIONS = 199, 199
CONFIGS, LOCAL_RUN, DRIVE_RUN = grid.setup(REPO_DIR, DRIVE_RESULTS,
    MODELS, HORIZONS, SEEDS, DEVICE, smoke=SMOKE_TEST)
print('Output:', DRIVE_RUN)
print('Mode/seed jobs:', len(CONFIGS)*len(SEEDS)*4)
print(json.dumps(next(iter(CONFIGS.values())).to_dict(), indent=2))


## 5. Train in seed-major order

Every completed job must contain all patients, predictions, metrics and histories;
its file hashes are checked before reuse. After a disconnect, rerun preceding cells
with the same code/config/environment to restore the same fingerprint.
Epoch logs are printed below. TL is fitted only once per cell/seed.


In [ ]:
grid.train(CONFIGS, LOCAL_RUN, DRIVE_RUN)

## 6. Verify paired trajectories, summarize sensitivity, and build analysis parents

All RL arms must share their validation-loss trajectory before their stopping policies
diverge. The sensitivity CSV reports MAE and RMSE with the same patient/seed bootstrap
draws across every comparison. BH correction spans all model × horizon × arm × metric
tests (72 with default settings). Change intervals are pointwise and exploratory;
containing zero is not proof of equivalence.

The standard analysis folders each pair an RL arm with the identical TL reference.
TL prediction folders are copied to keep every analysis parent self-contained in Drive.


In [ ]:
table = grid.summarize(CONFIGS, LOCAL_RUN, DRIVE_RUN, REPLICATES)
grid.merge(CONFIGS, LOCAL_RUN, DRIVE_RUN)
from IPython.display import display
display(table)
print('Sensitivity:', DRIVE_RUN / 'sensitivity')


## 7. Dataset analysis and full per-arm analysis

The following cells execute `RUN/run_dataset_analysis.sh` and every stage of
`RUN/run_experiments_analysis.sh`, separately for all three RL arms. They produce
cohort features, zone-D uncertainty, glycemic-range and rapid-change summaries,
patient figures, t-SNE, persistence, shift/screens, transfer inference, patient
stability, horizon tests and model comparisons. Prediction files remain available
inside every paired experiment folder.

A successful stage is saved to Drive. Stage outputs are hash-checked before skipping
on a restart. A failed stage raises an error and remains incomplete. Smoke runs skip
these cohort analyses because two patients are not sufficient for their inference.

Per-arm pipeline q-values have narrower families than the combined sensitivity CSV;
do not interpret the repeated TL outputs as independent experiments or combine claims
across arms without the broader correction.


### dataset

In [ ]:
if SMOKE_TEST:
    print('Skipping full-cohort analysis in smoke mode.')
else:
    for arm in grid.ARMS:
        grid.analyze_stage(REPO_DIR, CONFIGS, LOCAL_RUN, DRIVE_RUN, arm, 'dataset',
                           REPLICATES, PERMUTATIONS)


### zone-d

In [ ]:
if SMOKE_TEST:
    print('Skipping full-cohort analysis in smoke mode.')
else:
    for arm in grid.ARMS:
        grid.analyze_stage(REPO_DIR, CONFIGS, LOCAL_RUN, DRIVE_RUN, arm, 'zone-d',
                           REPLICATES, PERMUTATIONS)


### error-range

In [ ]:
if SMOKE_TEST:
    print('Skipping full-cohort analysis in smoke mode.')
else:
    for arm in grid.ARMS:
        grid.analyze_stage(REPO_DIR, CONFIGS, LOCAL_RUN, DRIVE_RUN, arm, 'error-range',
                           REPLICATES, PERMUTATIONS)


### rapid-change

In [ ]:
if SMOKE_TEST:
    print('Skipping full-cohort analysis in smoke mode.')
else:
    for arm in grid.ARMS:
        grid.analyze_stage(REPO_DIR, CONFIGS, LOCAL_RUN, DRIVE_RUN, arm, 'rapid-change',
                           REPLICATES, PERMUTATIONS)


### figures

In [ ]:
if SMOKE_TEST:
    print('Skipping full-cohort analysis in smoke mode.')
else:
    for arm in grid.ARMS:
        grid.analyze_stage(REPO_DIR, CONFIGS, LOCAL_RUN, DRIVE_RUN, arm, 'figures',
                           REPLICATES, PERMUTATIONS)


### glycemic

In [ ]:
if SMOKE_TEST:
    print('Skipping full-cohort analysis in smoke mode.')
else:
    for arm in grid.ARMS:
        grid.analyze_stage(REPO_DIR, CONFIGS, LOCAL_RUN, DRIVE_RUN, arm, 'glycemic',
                           REPLICATES, PERMUTATIONS)


### tsne

In [ ]:
if SMOKE_TEST:
    print('Skipping full-cohort analysis in smoke mode.')
else:
    for arm in grid.ARMS:
        grid.analyze_stage(REPO_DIR, CONFIGS, LOCAL_RUN, DRIVE_RUN, arm, 'tsne',
                           REPLICATES, PERMUTATIONS)


### persistence

In [ ]:
if SMOKE_TEST:
    print('Skipping full-cohort analysis in smoke mode.')
else:
    for arm in grid.ARMS:
        grid.analyze_stage(REPO_DIR, CONFIGS, LOCAL_RUN, DRIVE_RUN, arm, 'persistence',
                           REPLICATES, PERMUTATIONS)


### shift

In [ ]:
if SMOKE_TEST:
    print('Skipping full-cohort analysis in smoke mode.')
else:
    for arm in grid.ARMS:
        grid.analyze_stage(REPO_DIR, CONFIGS, LOCAL_RUN, DRIVE_RUN, arm, 'shift',
                           REPLICATES, PERMUTATIONS)


### transfer

In [ ]:
if SMOKE_TEST:
    print('Skipping full-cohort analysis in smoke mode.')
else:
    for arm in grid.ARMS:
        grid.analyze_stage(REPO_DIR, CONFIGS, LOCAL_RUN, DRIVE_RUN, arm, 'transfer',
                           REPLICATES, PERMUTATIONS)


### stability

In [ ]:
if SMOKE_TEST:
    print('Skipping full-cohort analysis in smoke mode.')
else:
    for arm in grid.ARMS:
        grid.analyze_stage(REPO_DIR, CONFIGS, LOCAL_RUN, DRIVE_RUN, arm, 'stability',
                           REPLICATES, PERMUTATIONS)


### horizons

In [ ]:
if SMOKE_TEST:
    print('Skipping full-cohort analysis in smoke mode.')
else:
    for arm in grid.ARMS:
        grid.analyze_stage(REPO_DIR, CONFIGS, LOCAL_RUN, DRIVE_RUN, arm, 'horizons',
                           REPLICATES, PERMUTATIONS)


### compare

In [ ]:
if SMOKE_TEST:
    print('Skipping full-cohort analysis in smoke mode.')
else:
    for arm in grid.ARMS:
        grid.analyze_stage(REPO_DIR, CONFIGS, LOCAL_RUN, DRIVE_RUN, arm, 'compare',
                           REPLICATES, PERMUTATIONS)


### summary

In [ ]:
if SMOKE_TEST:
    print('Skipping full-cohort analysis in smoke mode.')
else:
    for arm in grid.ARMS:
        grid.analyze_stage(REPO_DIR, CONFIGS, LOCAL_RUN, DRIVE_RUN, arm, 'summary',
                           REPLICATES, PERMUTATIONS)


## 8. Review sensitivity plots and outputs

`MyDrive/bg-results/cui_full_grid/<fingerprint>/` contains:

- `protocol.json`: exact configuration, source and input fingerprints.
- `jobs/`: original shared TL and three RL mode/seed jobs, predictions and histories.
- `paired/<arm>/`: analysis-ready model/horizon parents with all three seeds.
- `sensitivity/full_grid_sensitivity.csv`: all paired benefits and changes.
- `sensitivity/stopping_epochs.csv`: patient/seed stopping and best-validation epochs.
- `analysis/<arm>/`: all main-workflow analysis products.
- `logs/<arm>/`: parent lists and analysis logs.

Read absolute errors, persistence, paired changes and uncertainty together. Report all
arms and cells. The sensitivity answers the reviewer's stopping-policy concern within
this new protocol; it cannot isolate why the previously submitted numbers changed.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
table = pd.read_csv(LOCAL_RUN / 'sensitivity/full_grid_sensitivity.csv')
fig, axes = plt.subplots(len(MODELS), 1, figsize=(9, 3.5*len(MODELS)), squeeze=False, constrained_layout=True)
for ax, model in zip(axes[:,0], MODELS):
    for arm in grid.ARMS:
        t = table[(table.metric == 'mae') & (table.arm == arm) & table.cell.str.startswith(model+'_')].copy()
        t['horizon'] = t.cell.str.rsplit('_', n=1).str[-1].astype(int)
        t = t.sort_values('horizon')
        ax.plot(t.horizon, t.benefit_pct, marker='o', label=arm)
        ax.vlines(t.horizon, t.pct_low, t.pct_high, alpha=.4)
    ax.axhline(0, color='gray', lw=.8)
    ax.set(title=model.upper(), xlabel='Horizon (minutes)', ylabel='MAE reduction (%)')
    ax.legend()
fig.savefig(LOCAL_RUN / 'sensitivity/benefit_by_horizon.png', dpi=180)
plt.show()
grid.mirror(LOCAL_RUN / 'sensitivity', DRIVE_RUN / 'sensitivity')
print('All outputs:', DRIVE_RUN)
